# Milestone 3 · Notebook 01: Multi-Format Extraction & Format-Aware Chunking

**Objective:** Prototype and validate universal document extraction across all 6 supported corpus formats (`.pdf`, `.docx`, `.txt`, `.md`, `.pptx`, `.csv`), verify edge-case rejection (`empty_scanned.pdf`, `header_only.csv`, `spoofed_binary.pdf`), and compare **Config A (800 tokens / 120 overlap)** vs **Config B (400 tokens / 60 overlap + header-repeated CSV batching + contextual headers)** before freezing the production `ingestion/` module.

In [1]:
from __future__ import annotations

import csv
import io
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import chardet
import pandas as pd
import pdfplumber
import pymupdf
import tiktoken
from docx import Document as DocxDocument
from pptx import Presentation

# Resolve project root cleanly whether Jupyter launches from repo root or notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORPUS_DIR = PROJECT_ROOT / "data" / "corpus"
EDGE_DIR = PROJECT_ROOT / "data" / "edge_cases"

ENC = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """Return exact cl100k_base token count for a string."""
    return len(ENC.encode(text))


print(f"Project root : {PROJECT_ROOT}")
print(f"Corpus files : {sorted(p.name for p in CORPUS_DIR.iterdir() if p.is_file())}")
print(f"Edge cases   : {sorted(p.name for p in EDGE_DIR.iterdir() if p.is_file())}")

Project root : C:\Users\Abdul Wahab\Desktop\AI Portfolio-1\IEC-Portfolio1_RAG
Corpus files : ['01_company_hr_policy.pdf', '02_engineering_handbook.pdf', '03_remote_work_policy_2025.docx', '04_remote_work_addendum_2026.docx', '05_incident_runbook.md', '06_security_guidelines.txt', '07_q3_product_roadmap.pptx', '08_regional_sales_2025.csv']
Edge cases   : ['empty_scanned.pdf', 'header_only.csv', 'prompt_injection_doc.txt', 'spoofed_binary.pdf']


## 1. Canonical Intermediate Representation (`ExtractedBlock` & `ExtractedDocument`)

Every format extractor maps its native structure into a uniform list of `ExtractedBlock` records so downstream chunking, contextual header injection, and embedding remain 100% format-agnostic.

In [2]:
@dataclass
class ExtractedBlock:
    """Single normalized structural unit extracted from any supported document."""

    block_type: str  # "text" | "table" | "slide_notes" | "csv_table"
    content: str
    page_number: int = 1
    section_title: str = "General"
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class ExtractedDocument:
    """Canonical format-agnostic extraction output."""

    filename: str
    file_type: str
    page_count: int
    blocks: list[ExtractedBlock]


def _rows_to_markdown(headers: list[str], rows: list[list[str]]) -> str:
    """Serialize tabular headers and rows into a Markdown table."""
    clean_headers = [str(h or "").replace("|", "\\|").strip() for h in headers]
    lines = [
        "| " + " | ".join(clean_headers) + " |",
        "| " + " | ".join(["---"] * len(clean_headers)) + " |",
    ]
    for row in rows:
        padded = list(row) + [""] * max(0, len(clean_headers) - len(row))
        vals = [str(v or "").replace("|", "\\|").replace("\n", " ").strip() for v in padded[: len(clean_headers)]]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

## 2. Prototype Extractors for All 6 Formats (`.pdf`, `.docx`, `.txt`, `.md`, `.pptx`, `.csv`)

In [3]:
def extract_pdf(path: Path, min_chars_per_page: int = 20) -> ExtractedDocument:
    """Extract layout text via PyMuPDF and tables via pdfplumber, rejecting scanned PDFs."""
    raw = path.read_bytes()
    if not raw.startswith(b"%PDF-"):
        raise ValueError(f"415 Unsupported Media Type: {path.name} failed PDF magic-byte check.")

    blocks: list[ExtractedBlock] = []
    total_chars = 0
    with pymupdf.open(stream=raw, filetype="pdf") as pdf_doc:
        page_count = len(pdf_doc)
        for idx, page in enumerate(pdf_doc, start=1):
            # Strip repeating top/bottom running headers & footers via vertical clip rect
            rect = page.rect
            body_clip = pymupdf.Rect(rect.x0, rect.y0 + 36, rect.x1, rect.y1 - 36)
            text = page.get_text("text", clip=body_clip).strip()
            total_chars += len(text)
            if text:
                first_line = text.splitlines()[0].strip()[:120]
                blocks.append(
                    ExtractedBlock(
                        block_type="text",
                        content=text,
                        page_number=idx,
                        section_title=first_line,
                        metadata={"unit_type": "page"},
                    )
                )

    avg_chars = total_chars / max(page_count, 1)
    if avg_chars < min_chars_per_page:
        raise ValueError(
            f"400 Bad Request: {path.name} appears to be scanned/image-only "
            f"({avg_chars:.1f} chars/page < {min_chars_per_page}). OCR required."
        )

    # Extract explicit tables with pdfplumber when present
    with pdfplumber.open(io.BytesIO(raw)) as plumber_doc:
        for idx, page in enumerate(plumber_doc.pages, start=1):
            for tbl in page.extract_tables() or []:
                if len(tbl) >= 2:
                    md_tbl = _rows_to_markdown(tbl[0], tbl[1:])
                    blocks.append(
                        ExtractedBlock(
                            block_type="table",
                            content=md_tbl,
                            page_number=idx,
                            section_title=f"Table (Page {idx})",
                            metadata={"unit_type": "page"},
                        )
                    )

    return ExtractedDocument(path.name, "pdf", page_count, blocks)


def extract_docx(path: Path) -> ExtractedDocument:
    """Extract DOCX paragraphs & tables in reading order with heading tracking."""
    doc = DocxDocument(str(path))
    blocks: list[ExtractedBlock] = []
    current_section = "General"

    for p in doc.paragraphs:
        txt = p.text.strip()
        if not txt:
            continue
        style_name = (p.style.name or "").lower()
        if style_name.startswith("heading"):
            current_section = txt
        else:
            blocks.append(
                ExtractedBlock("text", txt, 1, current_section, {"unit_type": "section"})
            )

    for tbl in doc.tables:
        rows = [[cell.text.strip() for cell in row.cells] for row in tbl.rows]
        if len(rows) >= 2:
            blocks.append(
                ExtractedBlock(
                    "table",
                    _rows_to_markdown(rows[0], rows[1:]),
                    1,
                    current_section,
                    {"unit_type": "section"},
                )
            )

    return ExtractedDocument(path.name, "docx", 1, blocks)


def extract_text_or_md(path: Path) -> ExtractedDocument:
    """Decode TXT/MD via UTF-8/chardet; parse Markdown heading hierarchy."""
    raw = path.read_bytes()
    if b"\x00" in raw[:4096]:
        raise ValueError(f"415 Unsupported Media Type: {path.name} contains binary null bytes.")

    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        detected = chardet.detect(raw)["encoding"] or "latin-1"
        text = raw.decode(detected, errors="replace")

    text = text.replace("\r\n", "\n").strip()
    if not text:
        raise ValueError(f"400 Bad Request: {path.name} is empty.")

    ext = path.suffix.lower().lstrip(".")
    if ext == "txt":
        return ExtractedDocument(
            path.name, "txt", 1, [ExtractedBlock("text", text, 1, "Document Body", {"unit_type": "section"})]
        )

    # Markdown: split by heading lines while preserving breadcrumb hierarchy
    blocks: list[ExtractedBlock] = []
    heading_stack: list[tuple[int, str]] = []
    buf: list[str] = []

    def _flush() -> None:
        body = "\n".join(buf).strip()
        if body:
            crumb = " > ".join(h for _, h in heading_stack) or "Document Body"
            blocks.append(ExtractedBlock("text", body, 1, crumb, {"unit_type": "section"}))
        buf.clear()

    in_fence = False
    for line in text.splitlines():
        if line.strip().startswith("```"):
            in_fence = not in_fence
            buf.append(line)
            continue
        m = re.match(r"^(#{1,3})\s+(.+)$", line) if not in_fence else None
        if m:
            _flush()
            level = len(m.group(1))
            title = m.group(2).strip()
            heading_stack = [(lvl, t) for lvl, t in heading_stack if lvl < level]
            heading_stack.append((level, title))
        else:
            buf.append(line)
    _flush()

    return ExtractedDocument(path.name, "md", 1, blocks)


def extract_pptx(path: Path) -> ExtractedDocument:
    """Extract PPTX slide titles, text frames, tables, and speaker notes."""
    prs = Presentation(str(path))
    blocks: list[ExtractedBlock] = []

    for slide_num, slide in enumerate(prs.slides, start=1):
        title = slide.shapes.title.text.strip() if slide.shapes.title else f"Slide {slide_num}"
        text_parts: list[str] = []
        for shape in slide.shapes:
            if shape.has_text_frame and shape != slide.shapes.title:
                txt = shape.text_frame.text.strip()
                if txt:
                    text_parts.append(txt)
            if shape.has_table:
                tbl_rows = [[c.text.strip() for c in r.cells] for r in shape.table.rows]
                if len(tbl_rows) >= 2:
                    blocks.append(
                        ExtractedBlock(
                            "table",
                            _rows_to_markdown(tbl_rows[0], tbl_rows[1:]),
                            slide_num,
                            title,
                            {"unit_type": "slide"},
                        )
                    )
        if text_parts:
            blocks.append(
                ExtractedBlock("text", "\n".join(text_parts), slide_num, title, {"unit_type": "slide"})
            )
        if slide.has_notes_slide and slide.notes_slide.notes_text_frame:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                blocks.append(
                    ExtractedBlock("slide_notes", notes, slide_num, f"{title} (Notes)", {"unit_type": "slide"})
                )

    return ExtractedDocument(path.name, "pptx", len(prs.slides), blocks)


def extract_csv(path: Path) -> ExtractedDocument:
    """Sniff CSV encoding/delimiter and emit header + data rows for tabular batching."""
    raw = path.read_bytes()
    if b"\x00" in raw[:4096]:
        raise ValueError(f"415 Unsupported Media Type: {path.name} contains binary null bytes.")

    text = raw.decode("utf-8-sig")
    try:
        dialect = csv.Sniffer().sniff(text[:4096], delimiters=",;\t|")
    except csv.Error:
        dialect = csv.excel

    reader = list(csv.reader(io.StringIO(text), dialect))
    non_empty = [r for r in reader if any(cell.strip() for cell in r)]
    if len(non_empty) < 2:
        raise ValueError(f"400 Bad Request: {path.name} has no data rows (empty or header-only).")

    headers, data_rows = non_empty[0], non_empty[1:]
    return ExtractedDocument(
        path.name,
        "csv",
        1,
        [
            ExtractedBlock(
                block_type="csv_table",
                content="",
                page_number=1,
                section_title="CSV Dataset",
                metadata={"unit_type": "rows", "headers": headers, "rows": data_rows},
            )
        ],
    )


REGISTRY = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".txt": extract_text_or_md,
    ".md": extract_text_or_md,
    ".pptx": extract_pptx,
    ".csv": extract_csv,
}


# Validate extraction across the entire 8-file corpus
extracted_docs: list[ExtractedDocument] = []
summary_rows = []
for fpath in sorted(CORPUS_DIR.iterdir()):
    if fpath.is_file():
        doc = REGISTRY[fpath.suffix.lower()](fpath)
        extracted_docs.append(doc)
        summary_rows.append(
            {
                "filename": doc.filename,
                "format": doc.file_type,
                "pages_or_slides": doc.page_count,
                "blocks": len(doc.blocks),
            }
        )

display_df = pd.DataFrame(summary_rows)
print(display_df.to_string(index=False))

                         filename format  pages_or_slides  blocks
         01_company_hr_policy.pdf    pdf                2       2
      02_engineering_handbook.pdf    pdf                2       2
  03_remote_work_policy_2025.docx   docx                1       3
04_remote_work_addendum_2026.docx   docx                1       2
           05_incident_runbook.md     md                1       4
       06_security_guidelines.txt    txt                1       1
       07_q3_product_roadmap.pptx   pptx                4       8
       08_regional_sales_2025.csv    csv                1       1


## 3. Edge-Case Rejection Verification (`empty_scanned.pdf`, `header_only.csv`, `spoofed_binary.pdf`)

In [4]:
for edge_name in ["empty_scanned.pdf", "header_only.csv", "spoofed_binary.pdf"]:
    edge_path = EDGE_DIR / edge_name
    try:
        REGISTRY[edge_path.suffix.lower()](edge_path)
        print(f"[FAIL] {edge_name} was unexpectedly accepted!")
    except ValueError as exc:
        print(f"[PASS] {edge_name} rejected cleanly -> {exc}")

[PASS] empty_scanned.pdf rejected cleanly -> 400 Bad Request: empty_scanned.pdf appears to be scanned/image-only (0.0 chars/page < 20). OCR required.
[PASS] header_only.csv rejected cleanly -> 400 Bad Request: header_only.csv has no data rows (empty or header-only).
[PASS] spoofed_binary.pdf rejected cleanly -> 415 Unsupported Media Type: spoofed_binary.pdf failed PDF magic-byte check.


## 4. Format-Aware Chunking Comparison: Config A (Baseline) vs Config B (Optimized)

- **Config A**: `800` tokens, `120` overlap (15%), no contextual headers.
- **Config B**: `400` tokens, `60` overlap (15%), **header-repeated CSV/table row batching**, and **contextual metadata header injection** (`[Document | Section | Page/Slide/Rows]`).

In [5]:
def chunk_document(
    doc: ExtractedDocument,
    chunk_size: int = 400,
    chunk_overlap: int = 60,
    inject_headers: bool = True,
) -> list[dict[str, Any]]:
    """Partition an ExtractedDocument into token-bounded chunks."""
    chunks: list[dict[str, Any]] = []

    def _prefix(section: str, page: int, unit_type: str, extra: str = "") -> str:
        if not inject_headers:
            return ""
        unit_label = {"page": f"Page: {page}", "slide": f"Slide: {page}", "rows": extra}.get(
            unit_type, f"Section: {section}"
        )
        return f"[Document: {doc.filename} | Section: {section} | {unit_label}]\n"

    for block in doc.blocks:
        unit_type = block.metadata.get("unit_type", "section")

        # Tabular CSV: row-boundary batching with automatic header repetition
        if block.block_type == "csv_table":
            headers = block.metadata["headers"]
            data_rows = block.metadata["rows"]
            batch: list[list[str]] = []
            start_row = 1
            batch_idx = 1

            for idx, row in enumerate(data_rows, start=1):
                candidate_md = _rows_to_markdown(headers, [*batch, row])
                header_str = _prefix(
                    f"Rows {start_row}-{idx}",
                    batch_idx,
                    "rows",
                    f"Columns: {', '.join(headers)}",
                )
                if batch and count_tokens(header_str + candidate_md) > chunk_size:
                    final_text = (
                        _prefix(
                            f"Rows {start_row}-{idx - 1}",
                            batch_idx,
                            "rows",
                            f"Columns: {', '.join(headers)}",
                        )
                        + _rows_to_markdown(headers, batch)
                    )
                    chunks.append(
                        {
                            "document": doc.filename,
                            "file_type": doc.file_type,
                            "page_number": batch_idx,
                            "section_title": f"Rows {start_row}-{idx - 1}",
                            "content": final_text,
                            "token_count": count_tokens(final_text),
                        }
                    )
                    batch = [row]
                    start_row = idx
                    batch_idx += 1
                else:
                    batch.append(row)

            if batch:
                end_row = start_row + len(batch) - 1
                final_text = (
                    _prefix(
                        f"Rows {start_row}-{end_row}",
                        batch_idx,
                        "rows",
                        f"Columns: {', '.join(headers)}",
                    )
                    + _rows_to_markdown(headers, batch)
                )
                chunks.append(
                    {
                        "document": doc.filename,
                        "file_type": doc.file_type,
                        "page_number": batch_idx,
                        "section_title": f"Rows {start_row}-{end_row}",
                        "content": final_text,
                        "token_count": count_tokens(final_text),
                    }
                )
            continue

        # Narrative / Table blocks: recursive token-bounded split
        prefix = _prefix(block.section_title, block.page_number, unit_type)
        full_text = prefix + block.content
        if count_tokens(full_text) <= chunk_size or block.block_type == "table":
            chunks.append(
                {
                    "document": doc.filename,
                    "file_type": doc.file_type,
                    "page_number": block.page_number,
                    "section_title": block.section_title,
                    "content": full_text,
                    "token_count": count_tokens(full_text),
                }
            )
            continue

        # Token-window fallback with overlap for long narrative blocks
        tokens = ENC.encode(block.content)
        budget = max(chunk_size - count_tokens(prefix), 64)
        step = max(budget - chunk_overlap, 32)
        for start in range(0, len(tokens), step):
            sub_text = prefix + ENC.decode(tokens[start : start + budget])
            chunks.append(
                {
                    "document": doc.filename,
                    "file_type": doc.file_type,
                    "page_number": block.page_number,
                    "section_title": block.section_title,
                    "content": sub_text,
                    "token_count": count_tokens(sub_text),
                }
            )

    return chunks


chunks_a = [c for d in extracted_docs for c in chunk_document(d, 800, 120, inject_headers=False)]
chunks_b = [c for d in extracted_docs for c in chunk_document(d, 400, 60, inject_headers=True)]

for label, chks in [("Config A (800 tok, no headers)", chunks_a), ("Config B (400 tok, contextual headers)", chunks_b)]:
    toks = pd.Series([c["token_count"] for c in chks])
    print(
        f"{label:40s} | chunks={len(chks):2d} | min={toks.min():3d} | "
        f"median={int(toks.median()):3d} | p95={int(toks.quantile(0.95)):3d} | max={toks.max():3d}"
    )

print("\n--- Sample Config B Chunks (PDF, PPTX, CSV) ---")
for target_ext in ["pdf", "pptx", "csv"]:
    sample = next(c for c in chunks_b if c["file_type"] == target_ext)
    print(f"\n[{target_ext.upper()} · {sample['token_count']} tokens]\n{sample['content'][:360]}...")

Config A (800 tok, no headers)           | chunks=24 | min= 16 | median= 57 | p95=248 | max=788
Config B (400 tok, contextual headers)   | chunks=25 | min= 48 | median= 90 | p95=397 | max=397

--- Sample Config B Chunks (PDF, PPTX, CSV) ---

[PDF · 190 tokens]
[Document: 01_company_hr_policy.pdf | Section: ACME Cloud Corp · Employee Leave & Benefits Policy (HR-POL-2026) | Page: 1]
ACME Cloud Corp · Employee Leave & Benefits Policy (HR-POL-2026)
Section 1: Paid Annual Leave
Full-time employees are entitled to 22 days of paid annual leave per calendar year.
Up to 5 unused annual leave days may be carried over into Q...

[PPTX · 90 tokens]
[Document: 07_q3_product_roadmap.pptx | Section: Q3 2026 AI Platform Roadmap | Slide: 1]
Objective: Launch ACME Enterprise RAG v2.0 across 6 document formats.
Target Retrieval Quality: Hit Rate@5 >= 0.85 and MRR >= 0.75.
Target Faithfulness: >= 0.90 verified via automated RAGAS benchmarks....

[CSV · 397 tokens]
[Document: 08_regional_sales_2025.csv | S